# TimeR4 v11-FULL — Chạy TOÀN BỘ dữ liệu (4.054 câu test) trên Colab Pro

Bản chạy chính thức để lấy kết quả cuối cho khóa luận. Giống v11 (fine-tune chất
lượng cao: bf16, LoRA r=32, 2 epoch, MAX_LEN 1280) nhưng đánh giá trên **toàn bộ
tập test 4.054 câu**.

**Cơ chế an toàn cho lần chạy dài (~7-8h):**
1. **Lưu vào Google Drive** — adapter + kết quả lưu ở `MyDrive/tkgqa_outputs_v11full/`,
   sống sót khi runtime ngắt (khác `/content` bị xóa).
2. **Bỏ qua fine-tune nếu đã có adapter** — nếu bị ngắt rồi chạy lại, tự nạp adapter
   đã lưu, không train lại (~2h tiết kiệm).
3. **Lưu kết quả định kỳ mỗi 500 câu** — mất kết nối giữa chừng vẫn giữ được phần đã chạy.

> **Runtime:** GPU L4 (hoặc A100). **Nên dùng Colab Pro** để session đủ dài (tới 24h).
> Giữ tab mở hoặc dùng Colab Pro+ để chạy nền.


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 1 (Colab) — DRIVE + CÀI LIB + LOAD QWEN2.5-7B (4-bit, bf16) + LoRA r=32
# ═══════════════════════════════════════════════════════════
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'   # 1 GPU → tránh DataParallel
os.environ['WANDB_DISABLED'] = 'true'

# Mount Google Drive (nơi chứa dữ liệu). Bỏ qua nếu không phải Colab.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('✅ Đã mount Google Drive')
except Exception as _e:
    print('ℹ️ Không phải Colab hoặc mount lỗi:', str(_e)[:100])
os.environ['WANDB_MODE'] = 'disabled'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import subprocess, sys
subprocess.run([sys.executable,'-m','pip','uninstall','-y',
    'peft','sentence-transformers','transformers','huggingface-hub',
    'accelerate','wandb'], capture_output=True)

pkgs = [
    'huggingface_hub==0.23.4', 'peft==0.11.1',
    'transformers==4.44.0', 'sentence-transformers==3.3.1',
    'accelerate==0.34.0', 'faiss-cpu', 'tqdm',
]
for pkg in pkgs:
    r = subprocess.run([sys.executable,'-m','pip','install','-q',pkg], capture_output=True, text=True)
    print(f'  {"✅" if r.returncode==0 else "❌"} {pkg}')
    if r.returncode != 0: print('    ', r.stderr[-300:])

# bitsandbytes: nâng lên bản mới để khớp triton 3.x của Kaggle (0.43.1 lỗi 'triton.ops')
rb = subprocess.run([sys.executable,'-m','pip','install','-q','-U','bitsandbytes>=0.44'],
                    capture_output=True, text=True)
print(f'  {"✅" if rb.returncode==0 else "❌"} bitsandbytes (nâng cấp)')
if rb.returncode != 0: print('    ', rb.stderr[-300:])
try:
    import bitsandbytes as _bnb
    print(f'  ✅ bitsandbytes {_bnb.__version__} import OK')
except Exception as e:
    print('  ⚠️ bitsandbytes import lỗi:', str(e)[:200])
    print('     → thử: pip install -U bitsandbytes (Restart Session rồi chạy lại)')

import torch, json, glob, re, random, copy
from contextlib import nullcontext
from tqdm import tqdm
from collections import defaultdict
from sentence_transformers import SentenceTransformer
from transformers import (AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
                          TrainingArguments, Trainer)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
print('✅ Import OK')

assert torch.cuda.is_available(), '⚠️ Cần GPU! Runtime → Change runtime type → GPU (L4/A100)'
print(f'✅ GPU: {torch.cuda.get_device_name(0)}')
USE_BF16 = torch.cuda.is_bf16_supported()
_CDT = torch.bfloat16 if USE_BF16 else torch.float16
print(f'✅ bf16 hỗ trợ: {USE_BF16} → dùng {"bf16 (ổn định hơn)" if USE_BF16 else "fp16"}')
random.seed(42); torch.manual_seed(42)

MODEL_NAME = 'Qwen/Qwen2.5-7B-Instruct'
SYS_PROMPT = 'You are a precise temporal question-answering assistant.'

print(f'\nLoading {MODEL_NAME} (4-bit)... (~5-8 phút)')
bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=_CDT, bnb_4bit_use_double_quant=True)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

llm = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb,
    device_map={'': 0},                 # tất cả trên GPU0 cho ổn định khi train
    torch_dtype=_CDT, attn_implementation='eager')
llm.config.use_cache = False
# Dùng greedy decoding → gỡ tham số sampling để khỏi cảnh báo UserWarning
for _k in ('temperature','top_p','top_k'):
    if hasattr(llm.generation_config, _k):
        setattr(llm.generation_config, _k, None)
import warnings as _warnings
_warnings.filterwarnings('ignore', message='.*do_sample.*')
print(f'✅ Qwen2.5-7B (4-bit) loaded! VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB')

# ── LoRA ──
llm = prepare_model_for_kbit_training(llm)
lora_cfg = LoraConfig(
    r=32, lora_alpha=64, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'])
llm = get_peft_model(llm, lora_cfg)
llm.print_trainable_parameters()
DEVICE = 'cuda'


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 2 — CLONE REPO (cho retrival.py) + TKG (kg/full.txt) + 15 CSV CÂU HỎI
# ═══════════════════════════════════════════════════════════
import pandas as pd, ast

# Repo TimeR4 chỉ để lấy retrival.py (Cell 3 sẽ patch)
REPO_DIR = '/content/TimeR4'
if not os.path.exists(REPO_DIR):
    os.system(f'git clone --depth 1 https://github.com/qianxinying/TimeR4.git {REPO_DIR}')
os.chdir(REPO_DIR)

# Nơi tìm dữ liệu: Google Drive + /content (glob đệ quy, tên folder tùy ý)
DATA_ROOTS = ['/content/drive/MyDrive', '/content']

# ── TKG từ kg/full.txt (bộ Wikidata VI) ──
kg_cands = []
for _root in DATA_ROOTS:
    kg_cands += glob.glob(f'{_root}/**/kg/full.txt', recursive=True)
    kg_cands += glob.glob(f'{_root}/**/full.txt', recursive=True)
assert kg_cands, '❌ Không thấy full.txt — upload folder kg/ lên Google Drive (MyDrive/tkgqa_data/kg/)'
KG_PATH = kg_cands[0]
triple_list = []
with open(KG_PATH, encoding='utf-8') as f:
    for line in f:
        line = line.rstrip('\n')
        if not line.strip(): continue
        parts = line.split('\t')
        if len(parts) >= 4:
            triple_list.append([p.strip() for p in parts[:4]])
print(f'✅ TKG: {KG_PATH}')
print(f'   {len(triple_list):,} bộ tứ | ví dụ: {triple_list[0]}')

# ── 15 file câu hỏi CSV ──
q_files = []
for _root in DATA_ROOTS:
    q_files += glob.glob(f'{_root}/**/*_merged.csv', recursive=True)
q_files = sorted(set(q_files))
assert q_files, '❌ Không thấy *_merged.csv — upload folder question_generated/ lên Google Drive'
print(f'\nTìm thấy {len(q_files)} file câu hỏi')

def parse_answers(x):
    try:
        v = ast.literal_eval(x) if isinstance(x, str) else x
        return v if isinstance(v, list) else [str(v)]
    except Exception:
        return [str(x)]

rows = []
for qf in q_files:
    try:
        df = pd.read_csv(qf)
    except Exception as e:
        print('  ⚠️ lỗi đọc', qf, e); continue
    for _, r in df.iterrows():
        q = str(r.get('question', '')).strip()
        if not q or q.lower() == 'nan': continue
        rows.append({
            'question':    q,
            'answers':     parse_answers(r.get('answers', '[]')),
            'qtype':       str(r.get('qtype', '')),
            'time_level':  str(r.get('time_level', '')),
            'answer_type': str(r.get('answer_type', '')),
        })
print(f'Tổng câu (thô): {len(rows):,}')

# ── Khử trùng lặp câu hỏi (tránh rò rỉ train/test) ──
seen, uniq = set(), []
for r in rows:
    if r['question'] in seen: continue
    seen.add(r['question']); uniq.append(r)
print(f'Sau khử trùng lặp: {len(uniq):,}')

# ── Chia train/test 80/20 (seed 42) ──
random.seed(42); random.shuffle(uniq)
split = int(len(uniq) * 0.8)
train_questions = uniq[:split]
test_questions  = uniq[split:]
_train_set = {r['question'] for r in train_questions}
_overlap = sum(1 for r in test_questions if r['question'] in _train_set)
print(f'✅ Train: {len(train_questions):,} | Test: {len(test_questions):,} | Chồng lấn: {_overlap}')
# thống kê loại câu hỏi
import collections as _col
_qt = _col.Counter(r['qtype'] for r in test_questions)
print('   qtype trong test (top):', dict(_qt.most_common(6)))


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 3 — TEN + PATCH retrival.py + RERANK v7 + LOAD RETRIEVER
# ═══════════════════════════════════════════════════════════

def normalize_temporal_expression(question):
    text = question.lower()
    spans = []
    for m in re.finditer(r'ngày\s+(\d{1,2})\s+tháng\s+(\d{1,2})\s+năm\s+(\d{4})', text):
        spans.append((m.start(), m.end(), f'{m.group(3)}-{int(m.group(2)):02d}-{int(m.group(1)):02d}'))
    for m in re.finditer(r'ngày\s+(\d{1,2})/(\d{1,2})/(\d{4})', text):
        spans.append((m.start(), m.end(), f'{m.group(3)}-{int(m.group(2)):02d}-{int(m.group(1)):02d}'))
    for m in re.finditer(r'tháng\s+(\d{1,2})\s+năm\s+(\d{4})', text):
        spans.append((m.start(), m.end(), f'{m.group(2)}-{int(m.group(1)):02d}'))
    for m in re.finditer(r'tháng\s+(\d{1,2})/(\d{4})', text):
        spans.append((m.start(), m.end(), f'{m.group(2)}-{int(m.group(1)):02d}'))
    if not spans: return text
    kept, last_end = [], -1
    for s, e, r in sorted(spans, key=lambda x: -(x[1]-x[0])):
        if s >= last_end:
            kept.append((s, e, r)); last_end = e
    kept.sort(key=lambda x: x[0])
    for s, e, r in reversed(kept):
        text = text[:s] + r + text[e:]
    return text

print('✅ Module TEN loaded')

with open('retrival.py') as f: code = f.read()

patches = [
    ('self.triplet_id_list = [[triple[0], triple[1], triple[2], triple[3]] for triple in id_list]',
     'self.triplet_id_list = [[triple[0], triple[1], triple[2], triple[3]] for triple in id_list] if id_list is not None else []'),
    ('self.model = SentenceTransformer(model_name, device="cuda")\n        self.embedding_size = embedding_size',
     'self.model = SentenceTransformer(model_name, device="cuda")\n        self.embedding_size = self.model.get_sentence_embedding_dimension()'),
    ('self.full_time = [datetime.strptime(triple[3], "%Y-%m-%d").date() for triple in triple_list]',
     'self.full_time = []\n        for triple in triple_list:\n            t = triple[3] if len(triple) > 3 else "2000-01-01"\n            if len(t) == 4: t += "-01-01"\n            elif len(t) == 7: t += "-01"\n            try: self.full_time.append(datetime.strptime(t[:10], "%Y-%m-%d").date())\n            except: self.full_time.append(datetime.strptime("2000-01-01", "%Y-%m-%d").date())'),
    ('corpus_embeddings = corpus_embeddings / np.linalg.norm(corpus_embeddings, axis=1)[:, None]',
     'if corpus_embeddings.ndim == 1:\n            corpus_embeddings = corpus_embeddings[np.newaxis, :]\n        corpus_embeddings = corpus_embeddings / np.linalg.norm(corpus_embeddings, axis=1)[:, None]'),
]
for old, new in patches:
    if old in code: code = code.replace(old, new)
with open('retrival.py', 'w') as f: f.write(code)
if 'retrival' in sys.modules: del sys.modules['retrival']
from retrival import Retrieval
print('✅ retrival.py patched (4 lỗi kỹ thuật)')


# ── FIX #2: Time-filtering function (Equation 9) ──────────
# ── FIX #2 (v7): Rerank thời gian CHÍNH XÁC — port từ re_rank_single_result ──
from datetime import date as _date, timedelta as _td

def extract_timestamp_from_question(question):
    """Sau TEN: YYYY-MM-DD / YYYY-MM / YYYY. Chỉ chấp nhận tháng 01-12, ngày 01-31
    để KHÔNG bắt nhầm dải năm kiểu '2005-2010' thành '2005-20'."""
    m = re.search(r'(\d{4}(?:-(?:0[1-9]|1[0-2])(?:-(?:0[1-9]|[12]\d|3[01]))?)?)', question)
    return m.group(1) if m else None

# ───────── (GIỮ LẠI hàm CŨ để so sánh trong cell chẩn đoán) ─────────
def time_filter_score_old(tq_str, t_str):
    try:
        def to_date(s):
            s = s[:10]
            if len(s) == 4: s += '-01-01'
            elif len(s) == 7: s += '-01'
            return datetime.strptime(s, '%Y-%m-%d')
        diff = (to_date(tq_str) - to_date(t_str)).days
        return (1 - abs(diff)/3650.0) if diff > 0 else -100
    except Exception:
        return 0

def rerank_facts_old(question, facts, fact_triples_raw=None):
    tq = extract_timestamp_from_question(question)
    if not tq: return facts
    scored = []
    for fact in facts:
        m = re.search(r'in\s+(\d{4}(?:-\d{2}(?:-\d{2})?)?)\.', fact)
        scored.append((fact, time_filter_score_old(tq, m.group(1)) if m else 0))
    valid = [x for x in scored if x[1] != -100]
    pool = valid if valid else scored
    pool.sort(key=lambda x: -x[1])
    return [f for f, s in pool]

# ───────── (MỚI v7) keyword tiếng Việt + chấm điểm theo hướng ─────────
def vi_time_keyword(qnorm):
    """Fallback khi không có qtype: đoán (filt, order) từ text tiếng Việt."""
    ql = qnorm.lower()
    if any(k in ql for k in ['đầu tiên', 'lần đầu', 'sớm nhất']): return (None, 'first')
    if any(k in ql for k in ['cuối cùng', 'gần đây nhất', 'gần nhất', 'lần cuối', 'mới nhất']): return (None, 'last')
    if 'trước' in ql: return ('before', None)
    if 'sau'   in ql: return ('after', None)
    return ('in', None)

def qtype_to_kw(qt):
    """v9: chuyển cột qtype của dataset thành (filt, order).
    Xử lý cả loại ghép: before_last, after_first, before_first, after_last."""
    q = (qt or '').lower()
    filt  = 'before' if 'before' in q else ('after' if 'after' in q else ('in' if 'equal' in q else None))
    order = 'first'  if 'first'  in q else ('last'  if 'last'  in q else None)
    return None if (filt is None and order is None) else (filt, order)

def q_period(tq):
    """Trả (start, end) bao trùm độ chi tiết mốc thời gian. None nếu không hợp lệ."""
    try:
        tq = tq[:10]
        if len(tq) == 4:
            y = int(tq); return _date(y,1,1), _date(y,12,31)
        if len(tq) == 7:
            y, m = int(tq[:4]), int(tq[5:7])
            if not (1 <= m <= 12): return None
            nxt = _date(y+1,1,1) if m == 12 else _date(y,m+1,1)
            return _date(y,m,1), nxt - _td(days=1)
        if len(tq) == 10:
            return _date(int(tq[:4]),int(tq[5:7]),int(tq[8:10])), _date(int(tq[:4]),int(tq[5:7]),int(tq[8:10]))
    except Exception:
        return None
    return None

def fact_date(fact):
    try:
        m = re.search(r'in\s+(\d{4})-(\d{2})-(\d{2})', fact)
        if m: return _date(int(m.group(1)), int(m.group(2)), int(m.group(3)))
        m = re.search(r'in\s+(\d{4})-(\d{2})', fact)
        if m: return _date(int(m.group(1)), int(m.group(2)), 1)
        m = re.search(r'in\s+(\d{4})', fact)
        if m: return _date(int(m.group(1)), 1, 1)
    except Exception:
        return None
    return None

def _chrono(facts, reverse):
    d = [(f, fact_date(f)) for f in facts]
    wt = [x for x in d if x[1] is not None]
    wo = [x for x in d if x[1] is None]
    wt.sort(key=lambda x: x[1], reverse=reverse)
    return [f for f, _ in wt] + [f for f, _ in wo]

def rerank_facts(question, facts, fact_triples_raw=None, kw=None):
    """v9: lọc theo vùng (before/after/in) rồi sắp theo first/last.
    kw = (filt, order) từ qtype; nếu None → đoán từ text."""
    if kw is None:
        kw = vi_time_keyword(question)
    filt, order = kw
    tq  = extract_timestamp_from_question(question)
    per = q_period(tq) if tq else None

    ok, bad = facts, []
    # B1: lọc theo vùng thời gian (chỉ khi có mốc + hướng lọc)
    if per and filt in ('before', 'after', 'in'):
        s0, e0 = per; ok, bad = [], []
        for f in facts:
            fd = fact_date(f)
            if fd is None:
                ok.append(f); continue
            good = (s0 <= fd <= e0) if filt == 'in' else (fd < s0 if filt == 'before' else fd > e0)
            (ok if good else bad).append(f)

    # B2: sắp thứ tự thời gian nếu là first/last (chỉ trên phần hợp-vùng)
    if order in ('first', 'last'):
        ok = _chrono(ok, reverse=(order == 'last'))

    return ok + bad

print('✅ Rerank v7 (chính xác theo hướng trước/sau/vào + granularity) sẵn sàng')

def get_q(x): return x.get('question', x.get('Question', ''))
def get_qtype(x): return x.get('qtype', '')

questions_1000 = test_questions   # tập test đã tách 80/20
print(f'\nTập test: {len(questions_1000)} câu (Wikidata VI)')

# ── Load retriever NỀN đa ngôn ngữ (dùng cho cả dựng-data lẫn inference) ──
RETRIEVER_BASE = 'sentence-transformers/paraphrase-multilingual-mpnet-base-v2'
# Nếu đã có retriever fine-tune v6 trong input/working thì ưu tiên dùng (tốt hơn):
import glob as _glob
_cand = _glob.glob('/content/**/finetuned_retriever_v6', recursive=True) + \
        (['/content/models/finetuned_retriever_v6'] if os.path.exists('/content/models/finetuned_retriever_v6') else [])
RETRIEVER_PATH = _cand[0] if _cand else RETRIEVER_BASE
print(f'✅ Retriever dùng: {RETRIEVER_PATH}')


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 4 — PROMPT (Figure 5/6) + GEN theo CHAT TEMPLATE QWEN
# rewrite dùng BASE (tắt adapter) ; reasoning dùng adapter đã fine-tune
# ═══════════════════════════════════════════════════════════
def build_rewrite_prompt(fact, question):
    return ('Replace the temporal fact in questions with explicit timestamps '
        'from the provided facts or your knowledge without any explanation. '
        'If you are not sure about the answer, return the original questions.\n\n'
        'For instance, from the fact:\n'
        '"[Juan Carlos I, Praise or endorse, Vietnam, 2006-02-22]",\n'
        'We can modify the question:\n'
        '"After Vietnam, who was the first to praise Juan Carlos I?"\n'
        'to "After 2006-02-22, who was the first to praise Juan Carlos I?"\n\n'
        'Here is your turn:\n'
        f'Facts: {fact}\nQuestion: {question}')

def build_reasoning_prompt(facts, question):
    return ('Based on the historical facts, please answer the given question. '
        'Please keep the answer as simple as possible and return all the '
        'possible answers as a list.\n'
        f'Historical facts: {facts}\nQuestion: {question}')

def _qwen_gen(user_content, max_new_tokens=64, use_adapter=True, max_in=1500):
    msgs = [{'role':'system','content':SYS_PROMPT},
            {'role':'user','content':user_content}]
    text = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
    inp = tokenizer(text, return_tensors='pt', truncation=True, max_length=max_in).to(DEVICE)
    ctx = (llm.disable_adapter() if (not use_adapter and hasattr(llm,'disable_adapter'))
           else nullcontext())
    with torch.no_grad(), ctx:
        out = llm.generate(**inp, max_new_tokens=max_new_tokens, do_sample=False,
                           pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inp['input_ids'].shape[1]:], skip_special_tokens=True).strip()

# rewrite: hành vi gốc (tắt adapter); reasoning: adapter fine-tune
USE_FT = True   # cell run sẽ bật/tắt để so base vs fine-tuned
def call_llm(prompt, max_new_tokens=60):
    return _qwen_gen(prompt, max_new_tokens=max_new_tokens, use_adapter=False).split('\n')[0].strip()
def gen_answer(prompt, max_new_tokens=64):
    raw = _qwen_gen(prompt, max_new_tokens=max_new_tokens, use_adapter=USE_FT)
    return raw.split('\n')[0].strip()

def parse_list_answer(raw):
    m = re.findall(r"['\"]([^'\"]+)['\"]", raw)
    return m if m else [raw.strip('[]\'\" .')]

def get_gold(x): return x.get('answer', x.get('answers', x.get('Answer', x.get('answers_simple','?'))))

def gold_objects(item):
    g = get_gold(item)
    if isinstance(g, str):
        inner = re.findall(r"['\"]([^'\"]+)['\"]", g); return inner if inner else [g]
    return g if isinstance(g, list) else [g]

import string as _string
def _normalize_ans(s):
    """Khớp ĐÚNG metric của tác giả (predict_answer.py normalize):
    lowercase + bỏ dấu câu + bỏ mạo từ a/an/the + bỏ <pad> + gộp khoảng trắng.
    Bỏ dấu câu giúp gold ICEWS 'Foreign Affairs (South Korea)' -> 'foreign affairs south korea'."""
    s = str(s).lower()
    s = ''.join(ch for ch in s if ch not in set(_string.punctuation))
    s = re.sub(r'\b(a|an|the)\b', ' ', s)
    s = re.sub(r'\b(pad)\b', ' ', s)
    return ' '.join(s.split())

def evaluate_hit(pred_raw, gold):
    """Theo tác giả: chuẩn hoá rồi kiểm tra gold (chuẩn hoá) NẰM TRONG prediction."""
    pred_str = _normalize_ans(pred_raw)
    golds = gold if isinstance(gold, list) else [gold]
    for g in golds:
        ng = _normalize_ans(g)
        if ng and ng in pred_str:
            return True
    return False

print('✅ Prompt + Qwen chat-template gen sẵn sàng')
print('--- smoke test reasoning ---')
print(gen_answer(build_reasoning_prompt("['UK hasLeader David Cameron in 2012-05-01.']", 'Who led UK in 2012?')))


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 5 — DỰNG DỮ LIỆU FINE-TUNE REASONING (Equation 11)
# Cho mỗi câu train: TEN → retrieve(top-50) → rerank → top-8 facts
#   prompt = Reasoning prompt (Figure 6) ; target = gold answer dạng list
# ═══════════════════════════════════════════════════════════
assert train_questions is not None, 'Cần MultiTQ_vi_train.json'

N_TRAIN   = 4000     # số câu train dùng để fine-tune (giảm nếu thiếu giờ)
N_RETR    = 50
TOPK      = 8

def gold_list_str(item):
    g = get_gold(item)
    if isinstance(g, str):
        inner = re.findall(r"['\"]([^'\"]+)['\"]", g)
        g = inner if inner else [g]
    if not isinstance(g, list): g = [g]
    return '[' + ', '.join(f"'{str(x)}'" for x in g) + ']'

# lọc câu có đáp án rõ ràng + có biểu thức thời gian (giống phân phối test)
train_pool = [q for q in train_questions
              if get_gold(q) not in (None, '?', '', [])][:N_TRAIN*2]
random.shuffle(train_pool)
train_pool = train_pool[:N_TRAIN]
print(f'Dựng training data từ {len(train_pool)} câu train...')

q_tr  = [normalize_temporal_expression(get_q(q)) for q in train_pool]
r_tr  = Retrieval(RETRIEVER_PATH, q_tr, triple_list, None, None)
d_tr, c_tr = r_tr.compute_similarity(n=N_RETR)
res_tr = r_tr.get_result(d_tr, c_tr, q_tr, re_rank=False)

train_examples = []
for i, item in enumerate(tqdm(train_pool, desc='Build pairs')):
    facts = res_tr[i].get('fact') or []
    facts = rerank_facts(q_tr[i], facts, kw=qtype_to_kw(train_pool[i].get('qtype','')))[:TOPK]
    if not facts:
        continue
    prompt = build_reasoning_prompt(facts, q_tr[i])
    target = gold_list_str(item)
    train_examples.append({'prompt': prompt, 'target': target})

print(f'✅ {len(train_examples)} cặp training.')
print('\n--- VÍ DỤ ---')
print('PROMPT:', train_examples[0]['prompt'][:240], '...')
print('TARGET:', train_examples[0]['target'])

del r_tr; torch.cuda.empty_cache()


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 6 (v11-FULL) — QLoRA + LƯU DRIVE + SKIP nếu đã có adapter (bf16, r=32, 2 epoch, MAX_LEN 1280)
# ═══════════════════════════════════════════════════════════
MAX_LEN = 1280
EPOCHS  = 2

class SFTDataset(torch.utils.data.Dataset):
    def __init__(self, examples):
        self.rows = []
        for ex in examples:
            msgs = [{'role':'system','content':SYS_PROMPT},
                    {'role':'user','content':ex['prompt']}]
            prompt_ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, tokenize=True)
            target_ids = tokenizer(ex['target'] + tokenizer.eos_token, add_special_tokens=False)['input_ids']
            input_ids = (prompt_ids + target_ids)[:MAX_LEN]
            labels    = ([-100]*len(prompt_ids) + target_ids)[:MAX_LEN]
            if len(input_ids) <= len(prompt_ids):   # đáp án bị cắt hết → bỏ
                continue
            self.rows.append({'input_ids':input_ids,'labels':labels})
    def __len__(self): return len(self.rows)
    def __getitem__(self, i): return self.rows[i]

def collate(batch):
    maxlen = max(len(x['input_ids']) for x in batch)
    pad = tokenizer.pad_token_id
    ids, lab, am = [], [], []
    for x in batch:
        d = maxlen - len(x['input_ids'])
        ids.append(x['input_ids'] + [pad]*d)
        lab.append(x['labels'] + [-100]*d)
        am.append([1]*len(x['input_ids']) + [0]*d)
    return {'input_ids':torch.tensor(ids), 'labels':torch.tensor(lab),
            'attention_mask':torch.tensor(am)}

# Thư mục lưu BỀN VỮNG trên Google Drive (sống sót khi runtime ngắt)
OUT_DIR  = '/content/drive/MyDrive/tkgqa_outputs_v11full'
os.makedirs(OUT_DIR, exist_ok=True)
LORA_DIR = f'{OUT_DIR}/qwen_lora_reasoning_v11'

_adapter_exists = (os.path.exists(os.path.join(LORA_DIR, 'adapter_model.safetensors')) or
                   os.path.exists(os.path.join(LORA_DIR, 'adapter_model.bin')))
_loaded = False
if _adapter_exists:
    try:
        from peft import set_peft_model_state_dict
        try:    from peft import load_peft_weights
        except Exception: from peft.utils import load_peft_weights
        set_peft_model_state_dict(llm, load_peft_weights(LORA_DIR))
        print(f'✅ Đã có adapter tại {LORA_DIR} → NẠP LẠI, bỏ qua fine-tune (tiết kiệm ~2h)')
        _loaded = True
    except Exception as e:
        print('⚠️ Nạp adapter lỗi, sẽ fine-tune lại:', str(e)[:120])

if not _loaded:
    ds = SFTDataset(train_examples)
    print(f'Dataset: {len(ds)} mẫu (sau khi lọc cắt cụt)')
    args = TrainingArguments(
        output_dir='/content/qwen_lora_ckpt',
        per_device_train_batch_size=2, gradient_accumulation_steps=8,   # batch hiệu dụng 16
        num_train_epochs=EPOCHS, learning_rate=2e-4, warmup_ratio=0.03,
        logging_steps=20, save_strategy='no',
        bf16=USE_BF16, fp16=(not USE_BF16),          # bf16 trên L4/A100
        gradient_checkpointing=True, gradient_checkpointing_kwargs={'use_reentrant': False},
        optim='paged_adamw_8bit', report_to='none', lr_scheduler_type='cosine')
    trainer = Trainer(model=llm, args=args, train_dataset=ds, data_collator=collate)
    llm.config.use_cache = False
    print(f'🚀 Fine-tune: {EPOCHS} epoch, batch hiệu dụng 16, {"bf16" if USE_BF16 else "fp16"}, r=32...')
    trainer.train()
    llm.save_pretrained(LORA_DIR)
    print(f'✅ Đã lưu LoRA adapter vào Drive: {LORA_DIR}')
# QUAN TRỌNG: tắt gradient checkpointing + bật KV-cache cho inference NHANH
try: llm.gradient_checkpointing_disable()
except Exception: pass
llm.config.use_cache = True
llm.eval()


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 7 — PIPELINE v8 (giống v7, LLM = Qwen fine-tuned)
# ═══════════════════════════════════════════════════════════
def run_pipeline_v8(qs_raw, triples, retriever_path, n_retrieve=50, top_k_facts=8, label='', checkpoint_path=None):
    qs = copy.deepcopy(qs_raw)
    print(f'\n{"="*60}\n  {label}\n{"="*60}')
    q_list = [get_q(q) for q in qs]
    q_norm = [normalize_temporal_expression(q) for q in q_list]

    print('[1/4] Retrieve (FKS)...')
    r1 = Retrieval(retriever_path, q_norm, triples, None, None)
    d1, c1 = r1.compute_similarity(n=n_retrieve)
    bg = r1.get_result(d1, c1, q_norm, re_rank=False)

    print('[2/4] Rewrite (base, tắt adapter)...')
    for i in tqdm(range(len(qs)), desc='Rewrite', leave=False):
        q = get_q(qs[i]); f = (bg[i].get('fact') or [''])[0]
        resp = call_llm(build_rewrite_prompt(f, q))
        qs[i]['question'] = resp if resp else q

    print('[3/4] TA-Retrieve + Rerank...')
    q2 = [normalize_temporal_expression(get_q(q)) for q in qs]
    r2 = Retrieval(retriever_path, q2, triples, None, None)
    d2, c2 = r2.compute_similarity(n=n_retrieve)
    fl = r2.get_result(d2, c2, q2, re_rank=False)
    for i in range(len(fl)):
        fl[i]['fact'] = rerank_facts(q2[i], fl[i].get('fact') or [], kw=qtype_to_kw(get_qtype(qs[i])))

    print('[4/4] Reasoning (Qwen fine-tuned)...')
    results, correct = [], 0
    for i, item in enumerate(tqdm(qs, desc='Generate', leave=False)):
        facts = (fl[i].get('fact') or [])[:top_k_facts]
        gold  = get_gold(qs_raw[i])
        pred  = gen_answer(build_reasoning_prompt(facts, get_q(item)))
        ok = evaluate_hit(pred, gold)
        correct += ok
        results.append({'question':get_q(qs_raw[i]), 'q_norm':q_norm[i], 'gold':str(gold),
                        'predicted':pred, 'parsed':parse_list_answer(pred),
                        'correct':ok, 'retrieved_facts':facts})
        # 💾 lưu định kỳ để không mất khi runtime ngắt
        if checkpoint_path and (i + 1) % 500 == 0:
            with open(checkpoint_path, 'w', encoding='utf-8') as _cf:
                json.dump(results, _cf, ensure_ascii=False)
            print(f'    💾 checkpoint {i+1}/{len(qs)} câu (đúng {correct}) → Drive')
    em = correct/len(results)*100
    print(f'\n✅ {label}: {correct}/{len(results)} = {em:.2f}%')
    return results, em

print('✅ Pipeline v8 ready')


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 8 (FULL) — CHẠY TOÀN BỘ 4.054 CÂU + ablation, lưu Drive + checkpoint
# ═══════════════════════════════════════════════════════════
# ── Đảm bảo inference NHANH (tắt gradient checkpointing + bật KV-cache) ──
try: llm.gradient_checkpointing_disable()
except Exception: pass
llm.config.use_cache = True
llm.eval()
print('use_cache =', llm.config.use_cache, '(phải True → generate nhanh)')

# CHẠY TOÀN BỘ tập test
TEST_N      = len(test_questions)   # = 4.054 câu
ABLATION_N  = 200
print(f'Sẽ đánh giá TOÀN BỘ {TEST_N} câu test (~7-8h). Kết quả lưu vào Drive định kỳ.')
_CKPT = f'{OUT_DIR}/results_v11_full_ckpt.json'

# (A) Fine-tuned trên toàn bộ test
USE_FT = True
res_ft, em_ft = run_pipeline_v8(questions_1000[:TEST_N], triple_list, RETRIEVER_PATH,
                                label=f'Qwen2.5 FINE-TUNED (n={TEST_N})', checkpoint_path=_CKPT)
with open(f'{OUT_DIR}/results_v11_full.json','w',encoding='utf-8') as f:
    json.dump(res_ft, f, ensure_ascii=False, indent=2)
print(f'✅ Kết quả đầy đủ đã lưu: {OUT_DIR}/results_v11_full.json')

# (B) Base (chưa fine-tune) trên subset → đo delta do fine-tune
USE_FT = False
res_base, em_base = run_pipeline_v8(questions_1000[:ABLATION_N], triple_list, RETRIEVER_PATH,
                                    label=f'Qwen2.5 BASE (n={ABLATION_N}, ablation)')
USE_FT = True

# fine-tuned trên cùng subset để so công bằng
res_ft_sub = res_ft[:ABLATION_N]
em_ft_sub = sum(x['correct'] for x in res_ft_sub)/len(res_ft_sub)*100

print('\n' + '═'*56)
print('  ABLATION (cùng subset, cùng retriever/pipeline)')
print('═'*56)
print(f'  Qwen BASE       : {em_base:.2f}%')
print(f'  Qwen FINE-TUNED : {em_ft_sub:.2f}%')
print(f'  → Tác động fine-tune LLM: {em_ft_sub-em_base:+.2f} điểm')


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 9 — BẢNG SO SÁNH + LƯU SUMMARY
# ═══════════════════════════════════════════════════════════
print('═'*66)
print('  KẾT QUẢ v11 (Colab, fine-tune chất lượng cao) — BỘ DỮ LIỆU WIKIDATA TIẾNG VIỆT')
print('═'*66)
print(f'  {"Cấu hình":<44} {"Hit@1":>8}')
print('  ' + '-'*62)
print(f'  {"Qwen2.5 BASE (chưa fine-tune, subset)":<44} {em_base:>7.2f}%')
print(f'  {"Qwen2.5 FINE-TUNED (subset)":<44} {em_ft_sub:>7.2f}%')
print(f'  {"→ Tác động fine-tune LLM":<44} {em_ft_sub-em_base:>+7.2f}')
print(f'  {"Qwen2.5 FINE-TUNED (toàn bộ test)":<44} {em_ft:>7.2f}%')
print('═'*66)

summary = {
    'dataset': 'Wikidata-VI (kg/full.txt + question_generated)',
    'em_v11_qwen_ft': round(em_ft, 2),
    'em_v11_qwen_base_subset': round(em_base, 2),
    'em_v11_qwen_ft_subset': round(em_ft_sub, 2),
    'finetune_gain_subset': round(em_ft_sub - em_base, 2),
    'test_n': TEST_N,
    'n_triples': len(triple_list),
}
with open(f'{OUT_DIR}/summary_v11_full_FINAL.json','w',encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
print('✅ summary_v11_FINAL.json + results_v11.json saved → tab Output')
